In [3]:
%pip install -q duckdb huggingface_hub pandas
import duckdb
import os
from google.colab import userdata

# Authenticate using the Colab Secret you set up
hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Define our base table paths
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"{REL}/fact_content_daily_performance/**/*.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Unit of Analysis: One row = one content item per client (client_hash_id + content_hash_id), aggregated over specific time windows.

Time Window:

Feature Window: April 1, 2026 – April 30, 2026 (Past performance).

Target Window: May 1, 2026 – May 31, 2026 (Future outcome).

By strictly separating these windows, we eliminate temporal leakage.

In [6]:
# Verify the time partitions we are going to use exist and contain data
window_check = con.sql(f"""
    SELECT
        EXTRACT(MONTH FROM report_date) AS month,
        COUNT(*) AS daily_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{FACT_DAILY}')
    WHERE report_date >= '2026-04-01' AND report_date <= '2026-05-31'
    GROUP BY EXTRACT(MONTH FROM report_date)
    ORDER BY month
""").df()

display(window_check)

,month,daily_rows,min_date,max_date
0,4,10424730,2026-04-01,2026-04-30
1,5,11687376,2026-05-01,2026-05-31


Unit of Analysis: One row = one content item per client (client_hash_id + content_hash_id), aggregated over specific time windows.

Time Window:

Feature Window: April 1, 2026 – April 30, 2026 (Past performance).

Target Window: May 1, 2026 – May 31, 2026 (Future outcome).

By strictly separating these windows, we eliminate temporal leakage.

In [4]:
# Build the foundational contract table in DuckDB to separate features and labels
contract_query = f"""
CREATE OR REPLACE TEMP TABLE my_contract_data AS
WITH april_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april,
        SUM(gsc_clicks) AS clicks_april,
        AVG(gsc_avg_position) AS avg_position_april
    FROM read_parquet('{FACT_DAILY}')
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
may_target AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_may
    FROM read_parquet('{FACT_DAILY}')
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    a.client_hash_id,
    a.content_hash_id,
    a.impressions_april,
    a.clicks_april,
    a.avg_position_april,
    m.clicks_may,
    -- The Label: 1 if May clicks dropped by > 20% compared to April
    CASE WHEN m.clicks_may < (a.clicks_april * 0.8) THEN 1 ELSE 0 END AS is_declining_in_may
FROM april_features a
JOIN may_target m
  ON a.client_hash_id = m.client_hash_id
  AND a.content_hash_id = m.content_hash_id
WHERE a.impressions_april > 100; -- Minimum volume filter to remove noise
"""

con.execute(contract_query)
print("Contract table built successfully. We have completely isolated the feature window from the target window.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Contract table built successfully. We have completely isolated the feature window from the target window.


3. Verify it with queries (grain, counts, missing values, windows)
(Paste into the Code cell)

Python
# 1. Grain Check: Ensure one row per client/content pair
grain_check = con.sql("""
    SELECT client_hash_id, content_hash_id, COUNT(*) as c
    FROM my_contract_data
    GROUP BY client_hash_id, content_hash_id
    HAVING c > 1
""").df()

# 2. Count Check: Total actionable pages available for modeling
count_check = con.sql("SELECT COUNT(*) AS total_actionable_pages FROM my_contract_data").df()

# 3. Missingness Check: Are there any nulls in our aggregated features?
null_check = con.sql("""
    SELECT
        SUM(CASE WHEN impressions_april IS NULL THEN 1 ELSE 0 END) AS null_impressions,
        SUM(CASE WHEN avg_position_april IS NULL THEN 1 ELSE 0 END) AS null_positions
    FROM my_contract_data
""").df()

print("1. Grain Violations (Should be Empty):")
display(grain_check)
print("\n2. Total Rows (Our N):")
display(count_check)
print("\n3. Missing Values in Features:")
display(null_check)

In [5]:
# 1. Grain Check: Ensure one row per client/content pair
grain_check = con.sql("""
    SELECT client_hash_id, content_hash_id, COUNT(*) as c
    FROM my_contract_data
    GROUP BY client_hash_id, content_hash_id
    HAVING c > 1
""").df()

# 2. Count Check: Total actionable pages available for modeling
count_check = con.sql("SELECT COUNT(*) AS total_actionable_pages FROM my_contract_data").df()

# 3. Missingness Check: Are there any nulls in our aggregated features?
null_check = con.sql("""
    SELECT
        SUM(CASE WHEN impressions_april IS NULL THEN 1 ELSE 0 END) AS null_impressions,
        SUM(CASE WHEN avg_position_april IS NULL THEN 1 ELSE 0 END) AS null_positions
    FROM my_contract_data
""").df()

print("1. Grain Violations (Should be Empty):")
display(grain_check)
print("\n2. Total Rows (Our N):")
display(count_check)
print("\n3. Missing Values in Features:")
display(null_check)

1. Grain Violations (Should be Empty):


,client_hash_id,content_hash_id,c



2. Total Rows (Our N):


,total_actionable_pages
0,106307



3. Missing Values in Features:


,null_impressions,null_positions
0,0.0,0.0


Data Limits & Limitations:

Unbalanced History: Because clients onboarded at different times, some clients have zero ga4_sessions during April 2026 simply because their tracking hadn't started yet. We must explicitly filter by ga4_data_available IS TRUE when adding analytics features so we don't confuse "no tracking" with "zero engagement."

Window Overlaps: We cannot safely use the fact_content_query_90d table as a feature source for this specific task. Because it uses a rolling 90-day window, its timeframe overlaps directly with our May target window, creating massive temporal leakage.

Causality: This data can tell us a page is declining, but it cannot tell us why. We can observe correlations with low click-through rates, but we cannot prove that a specific content edit will guarantee a recovery.

Feature (Safe prior knowledge): impressions_april, clicks_april, avg_position_april. All are aggregated strictly from the April 2026 partition.

Label (Future outcome): is_declining_in_may. A binary flag set to 1 if clicks_may are less than 80% of clicks_april (a 20% drop).

Context: client_hash_id, content_hash_id. Used for joining and grouped train/test splits, never as training features.

Excluded: trend_direction, trend_pct, and any data from June 2026. trend_direction is excluded because it is a pre-calculated product decision that leaks the answer. June data is excluded to reserve it as a blind test set if needed.

In [7]:
# Build the foundational contract table in DuckDB to separate features and labels
contract_query = f"""
CREATE OR REPLACE TEMP TABLE my_contract_data AS
WITH april_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april,
        SUM(gsc_clicks) AS clicks_april,
        AVG(gsc_avg_position) AS avg_position_april
    FROM read_parquet('{FACT_DAILY}')
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
may_target AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_may
    FROM read_parquet('{FACT_DAILY}')
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    a.client_hash_id,
    a.content_hash_id,
    a.impressions_april,
    a.clicks_april,
    a.avg_position_april,
    m.clicks_may,
    -- The Label: 1 if May clicks dropped by > 20% compared to April
    CASE WHEN m.clicks_may < (a.clicks_april * 0.8) THEN 1 ELSE 0 END AS is_declining_in_may
FROM april_features a
JOIN may_target m
  ON a.client_hash_id = m.client_hash_id
  AND a.content_hash_id = m.content_hash_id
WHERE a.impressions_april > 100; -- Minimum volume filter to remove noise
"""

con.execute(contract_query)
print("Contract table built successfully. We have completely isolated the feature window from the target window.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Contract table built successfully. We have completely isolated the feature window from the target window.


3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# 1. Grain Check: Ensure one row per client/content pair
grain_check = con.sql("""
    SELECT client_hash_id, content_hash_id, COUNT(*) as c
    FROM my_contract_data
    GROUP BY client_hash_id, content_hash_id
    HAVING c > 1
""").df()

# 2. Count Check: Total actionable pages available for modeling
count_check = con.sql("SELECT COUNT(*) AS total_actionable_pages FROM my_contract_data").df()

# 3. Missingness Check: Are there any nulls in our aggregated features?
null_check = con.sql("""
    SELECT
        SUM(CASE WHEN impressions_april IS NULL THEN 1 ELSE 0 END) AS null_impressions,
        SUM(CASE WHEN avg_position_april IS NULL THEN 1 ELSE 0 END) AS null_positions
    FROM my_contract_data
""").df()

print("1. Grain Violations (Should be Empty):")
display(grain_check)
print("\n2. Total Rows (Our N):")
display(count_check)
print("\n3. Missing Values in Features:")
display(null_check)

1. Grain Violations (Should be Empty):


,client_hash_id,content_hash_id,c



2. Total Rows (Our N):


,total_actionable_pages
0,106307



3. Missing Values in Features:


,null_impressions,null_positions
0,0.0,0.0


Data Limits & Limitations:

Unbalanced History: Because clients onboarded at different times, some clients have zero ga4_sessions during April 2026 simply because their tracking hadn't started yet. We must explicitly filter by ga4_data_available IS TRUE when adding analytics features so we don't confuse "no tracking" with "zero engagement."

Window Overlaps: We cannot safely use the fact_content_query_90d table as a feature source for this specific task. Because it uses a rolling 90-day window, its timeframe overlaps directly with our May target window, creating massive temporal leakage.

Causality: This data can tell us a page is declining, but it cannot tell us why. We can observe correlations with low click-through rates, but we cannot prove that a specific content edit will guarantee a recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.